In [6]:
import sys
!"{sys.executable}" -m pip install google-generativeai

In [8]:
import pandas as pd
import numpy as np
import joblib

import google.generativeai as genai

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import warnings
warnings.filterwarnings('ignore')
pd.set_option("display.max_columns",None)


In [9]:
ml_model = joblib.load("skincare_model.pkl")
label_encoders = joblib.load("label_encoders.pkl")

print("✅ GlowGuide AI Loaded Successfully!")

✅ GlowGuide AI Loaded Successfully!


In [10]:
genai.configure(api_key="YOUR API KEY")

gemini_model = genai.GenerativeModel("gemini-flash-latest")

print("✅ Gemini Connected!")

✅ Gemini Connected!


In [11]:
import ipywidgets as widgets
from IPython.display import display, Markdown

title = widgets.HTML(
    value="""
    <h1 style='color:#E75480; text-align:center;'>
        ✨ GlowGuide AI
    </h1>
    <h3 style='text-align:center;'>
        AI Powered Personalized Skincare Recommendation System
    </h3>
    """
)

display(title)

# ----------------------------
# Mandatory Inputs
# ----------------------------

age = widgets.IntSlider(
    value=22,
    min=15,
    max=60,
    description="Age:"
)

skin_type = widgets.Dropdown(
    options=[
        "Combination",
        "Dry",
        "Normal",
        "Oily",
        "Sensitive"
    ],
    value="Normal",
    description="Skin:"
)

concern = widgets.Dropdown(
    options=[
        "Acne",
        "Dryness",
        "Dullness",
        "Fine Lines",
        "Large Pores",
        "No Major Concern",
        "Pigmentation",
        "Uneven Texture"
    ],
    value="No Major Concern",
    description="Concern:"
)

budget = widgets.Dropdown(
    options=[
        "Under ₹500",
        "₹500–1000",
        "₹1000–2000",
        "Above ₹2000"
    ],
    value="₹500–1000",
    description="Budget:"
)

season = widgets.Dropdown(
    options=[
        "Monsoon",
        "Spring",
        "Summer",
        "Winter"
    ],
    value="Summer",
    description="Season:"
)

display(age)
display(skin_type)
display(concern)
display(budget)
display(season)

# ----------------------------
# Optional Inputs
# ----------------------------

water = widgets.Dropdown(
    options=[
        "Less than 1 L",
        "1–2 L",
        "More than 2 L"
    ],
    value="1–2 L",
    description="Water:"
)

sleep = widgets.Dropdown(
    options=[
        "Less than 6 hrs",
        "6–8 hrs",
        "More than 8 hrs"
    ],
    value="6–8 hrs",
    description="Sleep:"
)

sunscreen = widgets.Dropdown(
    options=[
        "Never",
        "Sometimes",
        "Always"
    ],
    value="Sometimes",
    description="Sunscreen:"
)

makeup = widgets.Dropdown(
    options=[
        "Never",
        "Rarely",
        "Occasionally",
        "Daily"
    ],
    value="Rarely",
    description="Makeup:"
)

sensitive = widgets.Dropdown(
    options=[
        "No",
        "Yes"
    ],
    value="No",
    description="Sensitive:"
)

routine = widgets.Dropdown(
    options=[
        "No Routine",
        "Basic",
        "Regular",
        "Advanced"
    ],
    value="Basic",
    description="Routine:"
)

optional_box = widgets.Accordion(
    children=[
        widgets.VBox([
            water,
            sleep,
            sunscreen,
            makeup,
            sensitive,
            routine
        ])
    ]
)

optional_box.set_title(0, "Optional Inputs (Click Here)")

display(optional_box)

HTML(value="\n    <h1 style='color:#E75480; text-align:center;'>\n        ✨ GlowGuide AI\n    </h1>\n    <h3 s…

IntSlider(value=22, description='Age:', max=60, min=15)

Dropdown(description='Skin:', index=2, options=('Combination', 'Dry', 'Normal', 'Oily', 'Sensitive'), value='N…

Dropdown(description='Concern:', index=5, options=('Acne', 'Dryness', 'Dullness', 'Fine Lines', 'Large Pores',…

Dropdown(description='Budget:', index=1, options=('Under ₹500', '₹500–1000', '₹1000–2000', 'Above ₹2000'), val…

Dropdown(description='Season:', index=2, options=('Monsoon', 'Spring', 'Summer', 'Winter'), value='Summer')

Accordion(children=(VBox(children=(Dropdown(description='Water:', index=1, options=('Less than 1 L', '1–2 L', …

In [12]:
# Read Mandatory Inputs

user_age = age.value
user_skin = skin_type.value
user_concern = concern.value
user_budget = budget.value
user_season = season.value

# Read Optional Inputs

user_water = water.value
user_sleep = sleep.value
user_sunscreen = sunscreen.value
user_makeup = makeup.value
user_sensitive = sensitive.value
user_routine = routine.value

print("✅ User Inputs Captured Successfully!\n")

print(f"Age              : {user_age}")
print(f"Skin Type        : {user_skin}")
print(f"Concern          : {user_concern}")
print(f"Budget           : {user_budget}")
print(f"Season           : {user_season}")

print("\n----- Optional Inputs -----")

print(f"Water Intake     : {user_water}")
print(f"Sleep Duration   : {user_sleep}")
print(f"Sunscreen Usage  : {user_sunscreen}")
print(f"Makeup Usage     : {user_makeup}")
print(f"Sensitive Skin   : {user_sensitive}")
print(f"Current Routine  : {user_routine}")

✅ User Inputs Captured Successfully!

Age              : 22
Skin Type        : Normal
Concern          : No Major Concern
Budget           : ₹500–1000
Season           : Summer

----- Optional Inputs -----
Water Intake     : 1–2 L
Sleep Duration   : 6–8 hrs
Sunscreen Usage  : Sometimes
Makeup Usage     : Rarely
Sensitive Skin   : No
Current Routine  : Basic


In [13]:
import pandas as pd

# Encode all categorical inputs

encoded_input = pd.DataFrame({
    "Age": [user_age],
    "Skin_Type": [label_encoders["Skin_Type"].transform([user_skin])[0]],
    "Skin_Concern": [label_encoders["Skin_Concern"].transform([user_concern])[0]],
    "Budget": [label_encoders["Budget"].transform([user_budget])[0]],
    "Season": [label_encoders["Season"].transform([user_season])[0]],
    "Water_Intake": [label_encoders["Water_Intake"].transform([user_water])[0]],
    "Sleep_Duration": [label_encoders["Sleep_Duration"].transform([user_sleep])[0]],
    "Sunscreen_Usage": [label_encoders["Sunscreen_Usage"].transform([user_sunscreen])[0]],
    "Makeup_Usage": [label_encoders["Makeup_Usage"].transform([user_makeup])[0]],
    "Sensitive_Skin": [label_encoders["Sensitive_Skin"].transform([user_sensitive])[0]],
    "Current_Routine": [label_encoders["Current_Routine"].transform([user_routine])[0]]
})

print("✅ Encoded Input Data")

display(encoded_input)

✅ Encoded Input Data


,Age,Skin_Type,Skin_Concern,Budget,Season,Water_Intake,Sleep_Duration,Sunscreen_Usage,Makeup_Usage,Sensitive_Skin,Current_Routine
0,22,2,5,3,2,0,0,2,3,0,1


In [14]:
prediction = ml_model.predict(encoded_input)[0]

confidence = ml_model.predict_proba(encoded_input).max() * 100

predicted_routine = label_encoders["Routine_Category"].inverse_transform([prediction])[0]

print("Predicted Routine :", predicted_routine)
print(f"Confidence : {confidence:.2f}%")

Predicted Routine : Basic Maintenance
Confidence : 77.00%


In [15]:
# User-friendly routine names

routine_display = {
    "Basic Maintenance": "🌿 Daily Skin Maintenance Routine",
    "Acne Control": "🧴 Acne Recovery Routine",
    "Hydration Focus": "💧 Hydration & Barrier Repair Routine",
    "Brightening": "✨ Bright & Even Tone Routine",
    "Sensitive Care": "🌸 Sensitive Skin Protection Routine",
    "Anti-Aging": "🌙 Anti-Aging & Repair Routine"
}

display_name = routine_display.get(predicted_routine, predicted_routine)

print("="*50)
print("✨ GlowGuide AI Prediction")
print("="*50)

print(f"Recommended Routine : {display_name}")
print(f"Model Confidence    : {confidence:.2f}%")

✨ GlowGuide AI Prediction
Recommended Routine : 🌿 Daily Skin Maintenance Routine
Model Confidence    : 77.00%


In [16]:
prompt = f"""
You are GlowGuide AI, an intelligent skincare assistant.

Below is the user's profile.

----------------------------
USER PROFILE
----------------------------

Age : {user_age}

Skin Type : {user_skin}

Skin Concern : {user_concern}

Budget : {user_budget}

Season : {user_season}

Water Intake : {user_water}

Sleep Duration : {user_sleep}

Sunscreen Usage : {user_sunscreen}

Makeup Usage : {user_makeup}

Sensitive Skin : {user_sensitive}

Current Routine : {user_routine}

----------------------------
MACHINE LEARNING RESULT
----------------------------

Recommended Routine :

{display_name}

Model Confidence :

{confidence:.2f}%

----------------------------

Generate a beautiful skincare report.

Follow this format exactly.

# 🌞 Morning Routine

Give 4-5 bullet points.

# 🌙 Night Routine

Give 4-5 bullet points.

# 🧴 Ingredients to Look For

Give 5 ingredients with one-line explanation.

# 🚫 Ingredients to Avoid

Give 5 ingredients with reason.

# 🥗 Diet Recommendations

Give healthy food recommendations.

# 💧 Lifestyle Tips

Give practical daily habits.

# ⚠️ Important Note

Mention that skincare advice is general and a dermatologist should be consulted for persistent skin conditions.

Keep the language simple and professional.
Do not recommend prescription medicines.
"""

In [14]:
pip install gradio google-genai pandas joblib scikit-learn

I0726 16:25:35.006280 16313865 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(102, generation: 1)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
!python app.py

/opt/anaconda3/lib/python3.13/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.6.1. This might

In [ ]:
!pip install gradio

In [ ]:
!pip install --upgrade scikit-learn

In [12]:
pip install groq markdown ipython

I0726 16:42:26.902275 16330386 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(100, generation: 1)
I0726 16:42:26.902335 16330386 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(100, generation: 1)
I0726 16:42:26.902337 16330386 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(100, generation: 1)
I0726 16:42:26.902339 16330386 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(100, generation: 1)
I0726 16:42:26.902340 16330386 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(100, generation: 1)
I0726 16:42:26.902342 16330386 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(100, generation: 1)
I0726 16:42:26.902343 16330386 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(100, generation: 1)
I0726 16:42:26.902345 16330386 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(100, generation: 1)
Note: you may need to restart the kernel to use updated packages.


In [13]:
import os
from groq import Groq
from IPython.display import Markdown, display

# 1. Set your Groq API Key
GROQ_API_KEY = "YOUR API KEY"  

# Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)

# 2. Define user variables
user_age = 25
user_skin = "Combination"
user_concern = "Acne & Hyperpigmentation"
user_budget = "Moderate"
user_season = "Summer"

# Optional ML placeholders (if you are predicting a specific routine name)
display_name = "Gentle Brightening Routine"
confidence = 94.5

# 3. Construct the Prompt
prompt = f"""
Act as a professional dermatologist and skincare expert.
Provide a complete, structured skincare routine based on the following profile:

- Age: {user_age}
- Skin Type: {user_skin}
- Skin Concern: {user_concern}
- Budget: {user_budget}
- Season: {user_season}

Please structure your response with clear sections:
1. Morning Routine (AM)
2. Evening Routine (PM)
3. Key Active Ingredients to look for
4. Crucial Tips for this season
"""

# 4. Generate Content via Groq API
print("⏳ Generating skincare routine using Groq...")

completion = client.chat.completions.create(
    model="llama-3.3-70b-versatile",  # Fast & high-quality model
    messages=[
        {"role": "system", "content": "You are a helpful skincare assistant."},
        {"role": "user", "content": prompt},
    ],
    temperature=0.7,
)

ai_response_text = completion.choices[0].message.content

# 5. Display Clean Output
display(
    Markdown(f"""
# ✨ GlowGuide AI

## 👤 User Profile
- **Age:** {user_age}
- **Skin Type:** {user_skin}
- **Skin Concern:** {user_concern}
- **Budget:** {user_budget}
- **Season:** {user_season}

---

## 🤖 AI Prediction
**Recommended Routine:** {display_name}  
**Model Confidence:** {confidence:.1f}%

---

{ai_response_text}
""")
)

⏳ Generating skincare routine using Groq...



# ✨ GlowGuide AI

## 👤 User Profile
- **Age:** 25
- **Skin Type:** Combination
- **Skin Concern:** Acne & Hyperpigmentation
- **Budget:** Moderate
- **Season:** Summer

---

## 🤖 AI Prediction
**Recommended Routine:** Gentle Brightening Routine  
**Model Confidence:** 94.5%

---

As a professional dermatologist and skincare expert, I've created a personalized skincare routine for you based on your profile. Here's a structured plan to help you tackle acne and hyperpigmentation:

### 1. Morning Routine (AM)

To start your day with a fresh and protected complexion, follow these steps:

1. **Cleanse**: Begin with a gentle, non-comedogenic cleanser that effectively removes dirt without stripping your skin of its natural oils. Look for a cleanser containing salicylic acid (around 0.5-1%) to help control acne. Apply a small amount to damp skin, massage gently, and rinse with lukewarm water.
2. **Tone**: Use a toner that balances your skin's pH and prepares it for further products. Witch hazel or a toner with niacinamide can help reduce the appearance of pores and improve skin elasticity.
3. **Essence/Serum**: Apply a lightweight, oil-free essence or serum that contains vitamin C. Vitamin C is crucial for brightening the skin, reducing hyperpigmentation, and protecting against environmental stressors.
4. **Moisturize**: Apply a lightweight, oil-free moisturizer that won't clog pores. Look for a moisturizer labeled "non-comedogenic" or "oil-free" and contains hyaluronic acid to provide hydration without greasiness.
5. **Sunscreen**: Finish your morning routine with a broad-spectrum sunscreen that has an SPF of at least 30. Choose a physical sunscreen (zinc oxide or titanium dioxide) for its anti-inflammatory properties and ability to provide a physical barrier against UV rays.

### 2. Evening Routine (PM)

In the evening, focus on deep cleansing, exfoliation, and treating your skin concerns:

1. **Cleanse**: Use a gentle cleanser as in the morning, but consider a double cleanse if you've worn makeup or sunscreen throughout the day. First, use a micellar water or a makeup remover to dissolve makeup, then follow up with your regular cleanser.
2. **Exfoliate**: 2-3 times a week, use a chemical exfoliant containing alpha-hydroxy acids (AHAs) like glycolic acid or beta-hydroxy acids (BHAs) like salicylic acid to help unclog pores and reduce acne. Start with a lower concentration and gradually increase as your skin becomes more tolerant.
3. **Treat**: Apply a treatment product that targets your specific skin concerns. For acne and hyperpigmentation, look for a product containing niacinamide, which can help improve skin elasticity, reduce inflammation, and diminish the appearance of hyperpigmentation.
4. **Moisturize**: Apply a moisturizer similar to the one used in the morning, but you can opt for a slightly richer formula for nighttime. However, keep in mind that during summer, you may still prefer a lightweight moisturizer to avoid greasiness.
5. **Spot Treat**: If you have any active acne spots, apply a spot treatment containing sulfur or benzoyl peroxide to help dry out the acne and reduce its size.

### 3. Key Active Ingredients to look for

- **Salicylic Acid**: Helps to unclog pores, reduce acne, and exfoliate the skin.
- **Vitamin C**: Essential for brightening the skin, reducing hyperpigmentation, and acting as an antioxidant.
- **Niacinamide**: Improves skin elasticity, reduces inflammation, and diminishes the appearance of hyperpigmentation.
- **Hyaluronic Acid**: Provides hydration without greasiness, making it ideal for combination skin.
- **Glycolic Acid**: An AHA that helps in exfoliating the skin, improving skin texture, and reducing the appearance of fine lines and hyperpigmentation.

### 4. Crucial Tips for this season

- **Stay Hydrated**: Drinking plenty of water is crucial, especially during summer, to keep your skin hydrated from the inside out.
- **Sun Protection**: Never skip sunscreen, even on cloudy days. Reapply every two hours or immediately after swimming or sweating.
- **Lightweight Products**: Opt for lightweight, oil-free products to avoid clogging pores and to keep your skin feeling fresh and cool.
- **Exfoliate Wisely**: While exfoliation is important, be gentle and do not over-exfoliate, as this can lead to irritation and dryness, especially in combination skin.
- **Consult a Dermatologist**: If your acne or hyperpigmentation persists or worsens, consider consulting a dermatologist for personalized advice and treatment options.

Remember, consistency and patience are key. It may take a few weeks to start seeing improvements in your skin. Stay committed to your routine, and with time, you should notice a reduction in acne and hyperpigmentation, leading to a clearer, more even-toned complexion.


In [14]:
!pip install gradio groq

In [18]:
import os
import gradio as gr
from groq import Groq

# ------------------------------------------------------------------
# 1. API Setup
# ------------------------------------------------------------------
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "YOUR_GROQ_API_KEY_HERE")
client = Groq(api_key=GROQ_API_KEY)


# ------------------------------------------------------------------
# 2. Report Generation Function
# ------------------------------------------------------------------
def generate_skincare_routine(age, skin_type, concern, budget, season):
    prompt = f"""
    Act as a professional dermatologist and skincare expert.
    Provide a complete, structured, and customized skincare routine based on the following profile:

    - Age: {age}
    - Skin Type: {skin_type}
    - Primary Skin Concern: {concern}
    - Budget Level: {budget}
    - Current Season: {season}

    Please structure your response with clear sections:
    1. Morning Routine (AM)
    2. Evening Routine (PM)
    3. Key Active Ingredients to Look For
    4. Crucial Tips for the Current Season
    """

    try:
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "system",
                    "content": "You are an expert dermatologist providing actionable, structured skincare advice.",
                },
                {"role": "user", "content": prompt},
            ],
            temperature=0.7,
        )

        ai_response = completion.choices[0].message.content

        # Beautifully formatted markdown output
        final_markdown = f"""
# ✨ GlowGuide AI Recommendations

### 👤 Profile Summary
- **Age:** {age} | **Skin Type:** {skin_type}
- **Concern:** {concern} | **Budget:** {budget} | **Season:** {season}

---

{ai_response}
        """
        return final_markdown

    except Exception as e:
        return f"❌ **Error connecting to Groq API:** {str(e)}"


# ------------------------------------------------------------------
# 3. Chatbot Handler Function
# ------------------------------------------------------------------
def chat_with_dermatologist(message, history):
    # Format chat history for Groq API
    groq_messages = [
        {
            "role": "system",
            "content": (
                "You are GlowGuide AI, a helpful, friendly, and knowledgeable skincare assistant. "
                "Answer user questions accurately, suggest ingredients, or clarify skincare routines."
            ),
        }
    ]

    for user_msg, bot_msg in history:
        groq_messages.append({"role": "user", "content": user_msg})
        groq_messages.append({"role": "assistant", "content": bot_msg})

    groq_messages.append({"role": "user", "content": message})

    try:
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=groq_messages,
            temperature=0.7,
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"Sorry, I ran into an error: {str(e)}"


# ------------------------------------------------------------------
# 4. Gradio UI Design (Multi-Tab Layout)
# ------------------------------------------------------------------
with gr.Blocks(theme=gr.themes.Soft(), title="GlowGuide AI") as demo:
    gr.Markdown(
        """
        # ✨ GlowGuide AI: Personal Skincare Hub
        *Powered by Llama 3.3 on Groq*
        """
    )

    with gr.Tabs():
        # TAB 1: ROUTINE GENERATOR
        with gr.TabItem("📋 Skincare Report Generator"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 👤 User Profile")

                    age = gr.Slider(
                        minimum=13,
                        maximum=80,
                        value=25,
                        step=1,
                        label="Age",
                    )

                    skin_type = gr.Dropdown(
                        choices=[
                            "Oily",
                            "Dry",
                            "Combination",
                            "Normal",
                            "Sensitive",
                        ],
                        value="Combination",
                        label="Skin Type",
                    )

                    concern = gr.Dropdown(
                        choices=[
                            "Acne & Hyperpigmentation",
                            "Anti-Aging & Fine Lines",
                            "Dryness & Dehydration",
                            "Redness & Sensitivity",
                            "Uneven Texture & Pores",
                        ],
                        value="Acne & Hyperpigmentation",
                        label="Primary Concern",
                    )

                    budget = gr.Radio(
                        choices=["Budget-Friendly", "Moderate", "Luxury"],
                        value="Moderate",
                        label="Budget Level",
                    )

                    season = gr.Radio(
                        choices=["Spring", "Summer", "Autumn", "Winter"],
                        value="Summer",
                        label="Season",
                    )

                    submit_btn = gr.Button(
                        "✨ Generate Routine Report", variant="primary"
                    )

                with gr.Column(scale=2):
                    output_markdown = gr.Markdown(
                        value="Fill in your profile on the left and click **Generate Routine Report**."
                    )

            submit_btn.click(
                fn=generate_skincare_routine,
                inputs=[age, skin_type, concern, budget, season],
                outputs=[output_markdown],
            )

        # TAB 2: INTERACTIVE CHATBOT
        with gr.TabItem("💬 Skincare Assistant Q&A"):
            gr.Markdown(
                "### Have questions about your routine or specific products? Ask below!"
            )

            gr.ChatInterface(
                fn=chat_with_dermatologist,
                textbox=gr.Textbox(
                    placeholder="e.g. Can I use Vitamin C and Niacinamide together?",
                    container=False,
                    scale=7,
                ),
                examples=[
                    "What order should I apply my products in?",
                    "Can I use Salicylic Acid every day?",
                    "What are good sunscreen options for oily skin?",
                ],
            )

# Launch local app
if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [1]:
!pip install groq

In [4]:
import joblib

skincare_model = joblib.load("skincare_model.pkl")

print("Model loaded successfully")

Model loaded successfully


In [5]:
label_encoders = joblib.load("label_encoders.pkl")

print("Encoders loaded successfully")

Encoders loaded successfully
